<a href="https://colab.research.google.com/github/samerahayer/AAI2025/blob/main/Exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1: Prompt Chaining for a Customer Support AI

**Goal:** Build a multi-step customer-service flow where each prompt uses information produced earlier in the chain.

**Tools:** ChatGPT for prompt design/refinement; Python in Google Colab for the runnable demonstration.

## Testing and Iteration

### First prompt version
```text
Read the customer message and respond helpfully.
```

**What I changed:** The first version was too vague because it did not specify a role, output format, allowed categories, safety constraints, or how later steps should use earlier outputs. I revised it into four linked prompts with explicit constraints and structured outputs.

## Step 1 Prompt - Classify the Issue
```text
You are a customer support triage assistant. Analyze the customer's message and return exactly three labeled fields: Issue Category, Customer Tone, and Urgency. Choose the issue category from Billing, Account Access, Technical Problem, Shipping, or Other. Do not invent details that the customer did not provide.
```

**How it links:** This creates the structured classification used by Steps 2, 3, and 4.

## Step 2 Prompt - Gather Missing Information
```text
You are the information-gathering step in a customer support chain. Using the original customer message AND the Step 1 classification below, decide what information is still needed before the issue can be resolved. Ask no more than two short questions. Do not ask for passwords, full payment-card numbers, or other sensitive information. If enough information is already available, say 'No additional information needed.'
```

**How it links:** This step uses both the original message and Step 1 classification to decide what information is still needed.

## Step 3 Prompt - Propose a Solution
```text
You are a customer support resolution assistant. Using the original message, Step 1 classification, and Step 2 information request, propose a safe next-step solution. Use a friendly, professional tone. Acknowledge the customer's concern, explain the next action, and do not promise a refund or outcome before verification.
```

**How it links:** This step uses the original message plus the outputs from Steps 1 and 2 to produce a safe next action.

## Step 4 Prompt - Apply the Escalation Rule
```text
You are the escalation checker at the end of a customer support prompt chain. Review the issue classification and proposed solution. Return exactly: Escalate: Yes/No, Reason: <one sentence>. Escalate only when the issue involves unresolved duplicate/unauthorized charges, account security, repeated failed troubleshooting, or something requiring a specialist.
```

**How it links:** This final step uses the classification and proposed solution to decide whether a specialist is required.

In [ ]:
customer_message = "I was charged twice for my monthly subscription and I need this fixed as soon as possible."

# These strings are the prompts that would be sent to an AI model in a production workflow.
step1_prompt = """You are a customer support triage assistant. Analyze the customer's message and return exactly three labeled fields: Issue Category, Customer Tone, and Urgency. Choose the issue category from Billing, Account Access, Technical Problem, Shipping, or Other. Do not invent details that the customer did not provide."""
step2_prompt = """You are the information-gathering step in a customer support chain. Using the original customer message AND the Step 1 classification below, decide what information is still needed before the issue can be resolved. Ask no more than two short questions. Do not ask for passwords, full payment-card numbers, or other sensitive information. If enough information is already available, say 'No additional information needed.'"""
step3_prompt = """You are a customer support resolution assistant. Using the original message, Step 1 classification, and Step 2 information request, propose a safe next-step solution. Use a friendly, professional tone. Acknowledge the customer's concern, explain the next action, and do not promise a refund or outcome before verification."""
step4_prompt = """You are the escalation checker at the end of a customer support prompt chain. Review the issue classification and proposed solution. Return exactly: Escalate: Yes/No, Reason: <one sentence>. Escalate only when the issue involves unresolved duplicate/unauthorized charges, account security, repeated failed troubleshooting, or something requiring a specialist."""

# For a classroom notebook that runs without an API key, the functions below simulate
# the expected model outputs while preserving the same prompt-chain dependencies.
def classify_issue(message):
    lower = message.lower()
    category = "Billing" if any(x in lower for x in ["charged", "billing", "subscription"]) else "Other"
    tone = "Frustrated/concerned" if any(x in lower for x in ["as soon as possible", "need this fixed", "charged twice"]) else "Neutral"
    urgency = "High" if any(x in lower for x in ["as soon as possible", "urgent"]) else "Normal"
    return {"Issue Category": category, "Customer Tone": tone, "Urgency": urgency}

def gather_missing_info(message, classification):
    if classification["Issue Category"] == "Billing" and "charged twice" in message.lower():
        return [
            "Could you confirm the date of the duplicate charge?",
            "Do both charges show the same amount on your statement?"
        ]
    return ["Please share any additional details that may help us review the issue."]

def propose_solution(message, classification, questions):
    return (
        "I'm sorry you're dealing with a duplicate subscription charge. "
        "Once the transaction date and amounts are confirmed, the billing team can compare the two charges and determine whether one was processed in error. "
        "Because the charge must be verified first, I would avoid promising a refund before that review is complete."
    )

def escalation_check(classification, solution):
    needs_escalation = classification["Issue Category"] == "Billing" and "duplicate" in solution.lower()
    return {
        "Escalate": "Yes" if needs_escalation else "No",
        "Reason": "A duplicate charge should be reviewed by a billing specialist before any correction is promised." if needs_escalation else "The issue can be handled through the standard support flow."
    }

classification = classify_issue(customer_message)
questions = gather_missing_info(customer_message, classification)
solution = propose_solution(customer_message, classification, questions)
escalation = escalation_check(classification, solution)

print("STEP 1 - CLASSIFY ISSUE")
for key, value in classification.items():
    print(f"{key}: {value}")

print("\nSTEP 2 - GATHER MISSING INFORMATION")
for q in questions:
    print("-", q)

print("\nSTEP 3 - PROPOSE SOLUTION")
print(solution)

print("\nSTEP 4 - ESCALATION RULE")
print("Escalate:", escalation["Escalate"])
print("Reason:", escalation["Reason"])


STEP 1 - CLASSIFY ISSUE
Issue Category: Billing
Customer Tone: Frustrated/concerned
Urgency: High

STEP 2 - GATHER MISSING INFORMATION
- Could you confirm the date of the duplicate charge?
- Do both charges show the same amount on your statement?

STEP 3 - PROPOSE SOLUTION
I'm sorry you're dealing with a duplicate subscription charge. Once the transaction date and amounts are confirmed, the billing team can compare the two charges and determine whether one was processed in error. Because the charge must be verified first, I would avoid promising a refund before that review is complete.

STEP 4 - ESCALATION RULE
Escalate: Yes
Reason: A duplicate charge should be reviewed by a billing specialist before any correction is promised.


## Why this demonstrates prompt chaining
The workflow is not a single prompt with follow-up wording. Each step has a distinct job, and later steps depend on earlier outputs: classification → missing information → solution → escalation decision. The revised prompts also add constraints for tone, safety, output format, and what the assistant should avoid.